In [0]:
%pip install hyperopt

In [0]:
# Herramientas de Machine Learning de Spark (Clasificación)
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml.classification import RandomForestClassifier, GBTClassifier 
from pyspark.ml.evaluation import BinaryClassificationEvaluator 

# MLflow y Hyperopt (sin cambios)
import mlflow
from hyperopt import fmin, tpe, hp, SparkTrials, STATUS_OK

# Cargar los datasets que ya preparamos
training_set = spark.table("workspace.data.pedidos_distribucion_target_training_set")
validation_set = spark.table("workspace.data.pedidos_distribucion_target_validation_set")

# Configurar el experimento en MLflow
mlflow.set_experiment("/Users/nicolascristanchomurcia@gmail.com/pedidos_distribucion_target")

In [0]:
# Identificar columnas categóricas y numéricas (excluyendo IDs y el target)
# (Este código es el mismo que en la respuesta anterior)

# Identificar columnas categóricas y numéricas
categorical_cols = [field for (field, dataType) in training_set.dtypes if dataType == "string" and field != 'cliente_id']
numeric_cols = [field for (field, dataType) in training_set.dtypes if (dataType in ["double", "int"]) and field != 'target_prox_trx_digital']

# Etapas del Pipeline
indexers = [StringIndexer(inputCol=c, outputCol=f"{c}_indexed", handleInvalid="keep") for c in categorical_cols]
encoder = OneHotEncoder(inputCols=[f"{c}_indexed" for c in categorical_cols], outputCols=[f"{c}_ohe" for c in categorical_cols])
feature_cols = [f"{c}_ohe" for c in categorical_cols] + numeric_cols
assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")

preprocessing_pipeline = Pipeline(stages=indexers + [encoder, assembler])

In [0]:
search_space = hp.choice('model_choice', [
    {
        'model': GBTClassifier(featuresCol='features', labelCol='target_prox_trx_digital'), # <-- CAMBIO
        'params': {
            'maxDepth': hp.quniform('gbt_maxDepth', 2, 8, 1),
            'maxIter': hp.quniform('gbt_maxIter', 20, 100, 5)
        }
    },
    {
        'model': RandomForestClassifier(featuresCol='features', labelCol='target_prox_trx_digital'), # <-- CAMBIO
        'params': {
            'maxDepth': hp.quniform('rf_maxDepth', 2, 10, 1),
            'numTrees': hp.quniform('rf_numTrees', 10, 150, 5),
            'impurity': hp.choice('rf_impurity', ['gini', 'entropy'])
        }
    }
])

In [0]:
# Evaluador para medir el rendimiento de nuestro modelo de clasificación
evaluator = BinaryClassificationEvaluator(labelCol="target_prox_trx_digital", metricName="areaUnderROC") # <-- CAMBIO

def objective_function(model_config):
    with mlflow.start_run() as run:
        model = model_config['model']
        params = model_config['params']
        model.setParams(**params)
        
        mlflow.log_params(params)
        mlflow.log_param("model_type", model.__class__.__name__)

        full_pipeline = Pipeline(stages=[preprocessing_pipeline, model])
        trained_model = full_pipeline.fit(training_set)
        predictions = trained_model.transform(validation_set)
        
        # Evaluar el modelo
        areaUnderROC = evaluator.evaluate(predictions) # <-- CAMBIO
        
        # Registrar la métrica (AUROC) en MLflow
        mlflow.log_metric("areaUnderROC", areaUnderROC) # <-- CAMBIO
        mlflow.spark.log_model(trained_model, "spark-model")

    # Hyperopt MINIMIZA el loss, y nosotros queremos MAXIMIZAR el AUROC.
    # Por lo tanto, retornamos 1 - AUROC.
    return {'loss': 1 - areaUnderROC, 'status': STATUS_OK} # <-- CAMBIO CRÍTICO

In [0]:
# --- Setup: Carga y Preparación ---
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml.classification import GBTClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator

training_set = spark.table("workspace.data.pedidos_distribucion_target_training_set")
validation_set = spark.table("workspace.data.pedidos_distribucion_target_validation_set")

print("Paso 1: Definiendo el pipeline de preprocesamiento...")
# --- Pipeline de Preprocesamiento (sin cambios) ---
categorical_cols = [field for (field, dataType) in training_set.dtypes if dataType == "string" and field != 'cliente_id']
numeric_cols = [field for (field, dataType) in training_set.dtypes if (dataType in ["double", "int"]) and field != 'target_prox_trx_digital']

indexers = [StringIndexer(inputCol=c, outputCol=f"{c}_indexed", handleInvalid="keep") for c in categorical_cols]
encoder = OneHotEncoder(inputCols=[f"{c}_indexed" for c in categorical_cols], outputCols=[f"{c}_ohe" for c in categorical_cols])
assembler = VectorAssembler(inputCols=[f"{c}_ohe" for c in categorical_cols] + numeric_cols, outputCol="features")

preprocessing_pipeline = Pipeline(stages=indexers + [encoder, assembler])

print("Paso 2: Definiendo un único modelo...")
# --- Modelo Único (en lugar del bucle de Hyperopt) ---
gbt = GBTClassifier(featuresCol='features', labelCol='target_prox_trx_digital', maxDepth=5, maxIter=20)
full_pipeline = Pipeline(stages=[preprocessing_pipeline, gbt])

try:
    print("Paso 3: Entrenando el modelo. Esto puede tardar unos minutos...")
    # --- Entrenamiento ---
    trained_model = full_pipeline.fit(training_set)

    print("Paso 4: Realizando predicciones en el set de validación...")
    # --- Predicción ---
    predictions = trained_model.transform(validation_set)

    print("Paso 5: Evaluando el modelo...")
    # --- Evaluación ---
    evaluator = BinaryClassificationEvaluator(labelCol="target_prox_trx_digital", metricName="areaUnderROC")
    areaUnderROC = evaluator.evaluate(predictions)

    print("¡Éxito!")
    print(f"El modelo único se entrenó correctamente.")
    print(f"Área Bajo la Curva ROC (AUROC): {areaUnderROC}")

except Exception as e:
    print("El entrenamiento del modelo simple falló.")
    print("Error:", e)

In [0]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()
sc = spark.sparkContext
print(sc)
